# Citation Preference Estimation and Bootstrap Analysis

This notebook computes the country–country citation preference measures used in the [paper](https://arxiv.org/abs/2404.05861), including the bootstrap-based estimates of over- and under-recognition.

It requires the processed outputs generated by the data preparation notebook **1_GlobalCitationNetwork_dataprep**, in particular:

- `oa_countrycites_nosameorg.csv.gz` — country-level citation counts with author- and affiliation-level self-citations removed  
- `oa_countrycites_nosameorg_journal.csv.gz` — the same citations with the journal of the cited publication

Both files must be available locally before running this notebook. If these inputs are not present, please first run **1_GlobalCitationNetwork_dataprep** or download the archived processed data provided in the shared data folder.

The outputs of this notebook are used directly in the subsequent analysis and figure-generation steps.

Outputs (in `path2countrydata`):

- `oa_countrycites_nosameorg_auc.csv.gz` — national citation preferences (AUC and DeLong variance)
- `nationalcitation_bootstrap_u100_{year}.csv.gz` — the 100 bootstrap AUCs per country pair, one file per year
- `oa_countrycites_nosameorg_journalbootstrap_auc.csv.gz` — the journal bootstrap citation preferences (Fig. 4)


In [ ]:
import os
import pandas as pd
import numpy as np
import scipy.stats as spstats
from statsmodels.stats.multitest import multipletests

import pyscisci.all as pyscisci

# set this path to where the OpenAlex database is stored
path2openalex = '/Users/hgt6rn/Documents/DataSets/OpenAlex'

path2countrydata = "/Users/hgt6rn/Documents/DataSets/OpenAlex/precomputed_metrics"

In [ ]:
# the traditional citation preference measure

dataset = "oa_countrycites_nosameorg"  # other data processing pipelines make: "oa_countrycites", "oa_countrycites_noself"


countrycites = pd.read_csv(os.path.join(path2countrydata, dataset + ".csv.gz"))

country_auc_df = []
def country_auc(citedf):
    for cited_c in citedf['Country'].unique():
        x = citedf[citedf['Country'] == cited_c]['CountryCitations'].values
        y = citedf[citedf['Country'] != cited_c]['CountryCitations'].values
        auc, delongcov = pyscisci.fast_delong(x, y)
        cname, y = citedf.name
        country_auc_df.append([cname, y, cited_c, auc, delongcov, x.shape[0]])

countrycites.groupby(['CitingCountry', 'CitedYear'])[['Country', 'CountryCitations']].apply(country_auc)
country_auc_df = pd.DataFrame(country_auc_df, columns=['CitingCountry', 'CitedYear', 'CitedCountry', 'AUC', 'Cov', 'N'])

country_auc_df.to_csv(os.path.join(path2countrydata, dataset + '_auc.csv.gz'), compression='gzip', 
                                       index=False, header=True, mode='w')



## Journal bootstrap citation preference (Fig. 4)

The baseline of each cited country is resampled to match the journals of that country's publications:
for every citing country, cited country and year with at least 50 cited publications, 100 baseline
samples are drawn with the same number of publications per journal as the cited country, and the AUC is
computed for each sample. The three steps below write
`oa_countrycites_nosameorg_journalbootstrap_auc.csv.gz`, the file used by notebook 6.

In [ ]:
# step 1: the citations with the journal of each cited publication (from notebook 1)
countrycites = pd.read_csv(os.path.join(path2countrydata, dataset + '_journal.csv.gz'))
countrycites = countrycites.rename(columns={'CitedJournalId': 'JournalId'})

# the publications without a journal are dropped
countrycites = countrycites.dropna(subset=['JournalId']).reset_index(drop=True)
countrycites['JournalId'] = countrycites['JournalId'].astype(int)

In [ ]:
# step 2: the journal bootstrap AUC, one file per cited year (the years can be run in batches)

def country_journal_auc_year(citedf, cname, y):

    baseline_df = citedf.drop_duplicates(subset=['CitedPublicationId']).reset_index(drop=True)

    for cited_c in citedf['Country'].unique():

        focus_c_df = citedf[citedf['Country'] == cited_c]

        xs = focus_c_df['CountryCitations'].values

        if xs.shape[0] >= 50:

            # the number of publications of the cited country in each journal
            baseline_df['sample_count'] = np.nan
            journal_counts = focus_c_df['JournalId'].value_counts()
            baseline_df['sample_count'] = baseline_df['JournalId'].map(journal_counts)
            sampled_df = baseline_df.dropna(subset=['sample_count'])

            for isample in range(nsamples):
                sampled_df = sampled_df.sample(frac=1)
                sampled_df['cum_count'] = sampled_df.groupby('JournalId', sort=False).cumcount()

                ys = sampled_df[sampled_df['cum_count'] < sampled_df['sample_count']]['CountryCitations'].values

                auc = pyscisci.fast_auc(xs, ys)

                country_auc_df_year.append([cname, y, cited_c, auc, xs.shape[0], isample])


nsamples = 100
for focus_year in range(1990, 2018):

    countrycites_year = countrycites[countrycites['CitedYear'] == focus_year]

    country_auc_df_year = []

    for citing_c in countrycites_year['CitingCountry'].unique():
        country_journal_auc_year(countrycites_year[countrycites_year['CitingCountry'] == citing_c],
                                 cname=citing_c, y=focus_year)

    aucdf = pd.DataFrame(country_auc_df_year, columns=['CitingCountry', 'CitedYear', 'CitedCountry', 'AUC', 'N', 'isample'])

    aucdf.to_csv(os.path.join(path2countrydata, 'nationalcitation_bootstrap_u100_{}.csv.gz'.format(focus_year)),
                 compression='gzip', index=False, header=True)
    print(focus_year, aucdf.shape)

In [ ]:
# step 3: combine the years, and the mean, standard deviation and significance over the 100 samples

country_auc_df = pd.concat([pd.read_csv(os.path.join(path2countrydata, 'nationalcitation_bootstrap_u100_{}.csv.gz'.format(focus_year)))
                            for focus_year in range(1990, 2018)])

country_auc_df['STD'] = country_auc_df.groupby(['CitingCountry', 'CitedYear', 'CitedCountry'])['AUC'].transform('std')
country_auc_df['AUC'] = country_auc_df.groupby(['CitingCountry', 'CitedYear', 'CitedCountry'])['AUC'].transform('mean')

# one row per citing country, cited country and year (the isample column is that of the first sample)
country_auc_df.drop_duplicates(['CitingCountry', 'CitedYear', 'CitedCountry'], inplace=True)

country_auc_df['Zscore'] = (country_auc_df['AUC'] - 0.5) / country_auc_df['STD']

country_auc_df = country_auc_df[country_auc_df['N'] >= 50].reset_index(drop=True)
country_auc_df['pvalue'] = spstats.norm.sf(np.abs(country_auc_df['Zscore'].values))

# Holm-Sidak correction with alpha = 0.01 within each year
def reject_holm(pvalues):
    reject, pvalues, _, _ = multipletests(pvalues, alpha=0.01, method='hs', is_sorted=False, returnsorted=False)
    return reject
country_auc_df['Significant'] = country_auc_df.groupby('CitedYear')['pvalue'].transform(reject_holm)

country_auc_df.to_csv(os.path.join(path2countrydata, dataset + '_journalbootstrap_auc.csv.gz'),
                      index=False, compression='gzip', header=True)